<a href="https://colab.research.google.com/github/Kommmi/Qaos/blob/main/07_Kicked_Top_Large_n.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 07 — Quantum kicked top at large n: symmetry, MPS, and projected ensembles

**Model** (same convention as `gqs.dynamics`):
$$U_F=\exp\!\Big[-i\frac{\kappa}{2j}J_z^2\Big]\exp\!\Big[-i\frac{\pi}{2}J_y\Big],\qquad j=\frac n2,\qquad J_\alpha=\sum_{i=1}^n S^\alpha_i .$$

**Why notebook 06's approach doesn't carry over.** The Ising chain had nearest-neighbour gates, which suit an MPS. Here
$$J_z^2=\sum_{i,l}S^z_iS^z_l$$
couples *every pair* of qubits: $n(n-1)/2$ all-to-all $ZZ$ gates per kick. In a 1D MPS those need swap networks.

**What the kicked top offers instead: permutation symmetry.** $U_F$ uses only collective operators, so it commutes with every permutation of the qubits. A spin-coherent initial state is symmetric, so the state stays in the $(n+1)$-dimensional **symmetric (Dicke) subspace**:
$$|\psi\rangle=\sum_{k=0}^{n}c_k\,|D_k\rangle,\qquad |D_k\rangle=\binom nk^{-1/2}\!\!\sum_{\text{strings with }k\text{ ones}}|s_1\cdots s_n\rangle .$$
We evolve $n+1$ numbers instead of $2^n$: exact, with no truncation.

**Sections**
1. Validate the Dicke-subspace dynamics against the full-space Qaos code.
2. Exact projected ensemble from the Dicke amplitudes, validated against `states.py`.
3. Reproduce the paper's Wasserstein diagnostic (`Compare_distances_All`).
4. Entanglement is bounded by symmetry: exact MPS with $\chi\le n/2+1$, and MPS sampling at 100 qubits.
5. GQS at 500 qubits: regular vs. chaotic $\kappa$.
6. Wasserstein sensitivity vs. system size up to 500 qubits, including the even/odd (integer vs. half-integer spin) comparison.

In [1]:
import time
import numpy as np
import matplotlib.pyplot as plt

from gqs import symmetric as S
from gqs.states import Initial_state, Reduced_state_single_site
from gqs.dynamics import Hamiltonian_QK, floquet_operator_from_H
from gqs.mps import sample_projected_ensemble, bloch_vectors, fubini_study_matrix
try:
    import ot   # POT, for Wasserstein distances
except ImportError:
    ot = None

# Same parameters as notebooks 03 and 04
kappa, theta, phi, eps = 2.5, np.pi/2 + 0.5, np.pi/2, 0.2

ModuleNotFoundError: No module named 'gqs'

## 1. Validation: Dicke subspace vs. full Hilbert space

In the Dicke basis ($k$ = number of $|1\rangle$'s, so $J_z=n/2-k$):
$$J_z|D_k\rangle=\Big(\tfrac n2-k\Big)|D_k\rangle,\qquad J_+|j,m\rangle=\sqrt{j(j+1)-m(m+1)}\,|j,m+1\rangle,\qquad J_y=\frac{J_+-J_-}{2i}.$$
The coherent state $\big(\cos\tfrac\theta2|0\rangle+e^{i\phi}\sin\tfrac\theta2|1\rangle\big)^{\otimes n}$ has
$$c_k=\sqrt{\tbinom nk}\,\cos^{n-k}\tfrac\theta2\,\sin^{k}\tfrac\theta2\,e^{ik\phi}.$$

In [ ]:
n, N_kicks = 10, 50
H1, H2 = Hamiltonian_QK(tau=1.0, kappa=kappa, nqubit=n)
U_full = floquet_operator_from_H(H1, H2, tau=1.0)          # 1024 x 1024
U_dicke = S.floquet_operator(n, kappa)                     # 11 x 11

psi = Initial_state(n, theta, phi)
c = S.coherent_state(n, theta, phi)
print(f"initial state:  max |full - Dicke| = {np.abs(psi - S.to_full_state(c)).max():.1e}")
for _ in range(N_kicks):
    psi, c = U_full @ psi, U_dicke @ c
print(f"after {N_kicks} kicks: |<psi_full|psi_Dicke>|^2 = {abs(np.vdot(psi, S.to_full_state(c)))**2:.12f}")
print(f"matrix sizes: full {U_full.shape}, Dicke {U_dicke.shape}")

## 2. Exact projected ensemble from Dicke amplitudes

Keep one qubit and measure the other $n-1$ in $Z$. A string $z$ with $m$ ones overlaps only $|D_m\rangle$ (system qubit 0) and $|D_{m+1}\rangle$ (system qubit 1), so
$$|\tilde\psi_z\rangle=\Big(\tfrac{c_m}{\sqrt{\binom nm}},\ \tfrac{c_{m+1}}{\sqrt{\binom n{m+1}}}\Big).$$
**Every outcome with the same $m$ gives the same conditional state.** Grouping the $\binom{n-1}{m}$ outcomes of each class, and using $\binom{n-1}m/\binom nm=\frac{n-m}n$ and $\binom{n-1}m/\binom n{m+1}=\frac{m+1}n$:
$$|\psi_m\rangle\propto\Big(\sqrt{\tfrac{n-m}{n}}\,c_m,\ \sqrt{\tfrac{m+1}{n}}\,c_{m+1}\Big),\qquad w_m=\tfrac{n-m}{n}|c_m|^2+\tfrac{m+1}{n}|c_{m+1}|^2,\qquad m=0,\dots,n-1 .$$
So the GQS has **at most $n$ points**, compared with $2^{n-1}$ outcomes in the full-space calculation, and no large binomials appear. By symmetry the result is the same for every system site.

In [ ]:
chi_sym, w_sym = S.projected_ensemble(c)
m_of_z = np.array([bin(z).count("1") for z in range(2**(n - 1))])   # class of each outcome

for site in [0, 4, 9]:
    chi_S, lam_E = Reduced_state_single_site(2, n, site, psi)        # full-space, 512 outcomes
    w_from_full = np.bincount(m_of_z, weights=lam_E, minlength=n)    # group outcomes by class m
    keep = lam_E > 1e-14
    fid = np.abs(np.sum(chi_S[keep].conj() * chi_sym[m_of_z[keep]], axis=1))**2
    print(f"site {site}:  max |w_m - grouped lambda_E| = {np.abs(w_from_full - w_sym).max():.1e},"
          f"  min fidelity of conditional states = {fid.min():.12f}")

## 3. The paper's Wasserstein diagnostic, reproduced

`gqs.Gamma.Compare_distances_All` evolves a reference and a perturbed coherent state, $(\theta,\phi)$ and $(\theta+\varepsilon,\phi)$, and tracks $W_1(t)/W_1(0)$ between their single-qubit GQS. Below is the same quantity from Dicke amplitudes. It uses the same Fubini–Study cost as `Quantum_EMD`.

In [ ]:
def gqs_wasserstein(c_a, c_b, tol=1e-14):
    # W1 between the single-qubit GQS of two symmetric states (Fubini-Study cost, as in Quantum_EMD)
    chi_a, w_a = S.projected_ensemble(c_a); chi_b, w_b = S.projected_ensemble(c_b)
    ka, kb = w_a > tol, w_b > tol
    return ot.emd2(w_a[ka] / w_a[ka].sum(), w_b[kb] / w_b[kb].sum(),
                   fubini_study_matrix(chi_a[ka], chi_b[kb]))

def sensitivity_curve(n, kappa, N_kicks):
    # W1(t)/W1(0) for reference vs. perturbed initial coherent states
    U = S.floquet_operator(n, kappa)
    a, b = S.coherent_state(n, theta, phi), S.coherent_state(n, theta + eps, phi)
    W0, out = gqs_wasserstein(a, b), np.empty(N_kicks)
    for t in range(N_kicks):
        out[t] = gqs_wasserstein(a, b) / W0
        a, b = U @ a, U @ b
    return out

if ot is None:
    print("Install POT (pip install POT) for this section and Section 6.")
else:
    from gqs.Gamma import Compare_distances_All
    n_small, K = 6, 30
    H1, H2 = Hamiltonian_QK(tau=1.0, kappa=kappa, nqubit=n_small)
    U6 = floquet_operator_from_H(H1, H2, tau=1.0)
    _, D_local, _ = Compare_distances_All(U6, Initial_state(n_small, theta, phi),
                                          Initial_state(n_small, theta + eps, phi),
                                          n_chain=n_small, system_site=0, N_kicks=K)
    D_sym = sensitivity_curve(n_small, kappa, K)
    print(f"n={n_small}: max |Compare_distances_All - Dicke version| = {np.abs(D_local - D_sym).max():.1e}")

## 4. Entanglement is bounded by symmetry: exact MPS

A symmetric state has an exact MPS whose bond index counts the ones seen so far:
$$A^{s}_{[i]}[k,k+s]=1\ \ (i<n-1),\qquad A^{s}_{[n-1]}[k,0]=\frac{c_{k+s}}{\sqrt{\binom n{k+s}}} .$$
Across the cut after site $i$, the Schmidt rank is at most $\min(i+1,\,n-i-1)+1$, so
$$\chi\le\frac n2+1,\qquad S_{\text{half}}\le\log_2\!\Big(\frac n2+1\Big).$$
Compare a generic chaotic state, which needs $\chi\sim2^{n/2}$. Symmetry keeps the kicked top's entanglement only logarithmic in $n$. This is why the problem is tractable, and why an MPS would also work.

In [ ]:
n = 12
c12 = S.coherent_state(n, theta, phi); U12 = S.floquet_operator(n, kappa)
for _ in range(30):
    c12 = U12 @ c12
m12 = S.to_mps(c12)
print("contracted MPS vs Dicke state, max error:", np.abs(m12.to_statevector() - S.to_full_state(c12)).max())
m12 = S.to_mps(c12)
ranks = [int(np.sum(m12.schmidt_values(i) > 1e-10)) for i in range(n - 1)]
print("Schmidt ranks across each cut:", ranks)
print("bound min(i+1, n-i-1)+1:      ", [min(i + 1, n - i - 1) + 1 for i in range(n - 1)])

In [ ]:
# Half-chain entanglement over time at n = 100: regular (kappa = 0.5) vs chaotic (kappa = 2.5)
n, K = 100, 60
plt.figure(figsize=(5.5, 3.5))
for kap in [0.5, 2.5]:
    U = S.floquet_operator(n, kap); ck = S.coherent_state(n, theta, phi); S_t = []
    for _ in range(K):
        S_t.append(S.to_mps(ck).entropy(n // 2 - 1))
        ck = U @ ck
    plt.plot(S_t, label=fr"$\kappa$={kap}")
plt.axhline(np.log2(n / 2 + 1), color="k", ls="--", lw=1, label=r"bound $\log_2(n/2+1)$")
plt.xlabel("kicks"); plt.ylabel("half-chain entropy (bits)"); plt.title(f"n = {n}")
plt.legend(); plt.tight_layout(); plt.show()

**MPS sampling vs. exact symmetric formula at 100 qubits.** This uses `sample_projected_ensemble` from notebook 06. Every sample must reproduce the exact conditional state of its class $m$ and the exact Born probability $p_z=w_m/\binom{n-1}{m}$.

In [ ]:
from scipy.special import gammaln
n, M = 100, 4000
c100 = S.coherent_state(n, theta, phi); U100 = S.floquet_operator(n, kappa)
for _ in range(30):
    c100 = U100 @ c100
chi_ex, w_ex = S.projected_ensemble(c100)

t0 = time.perf_counter()
chi_s, p_s, bits = sample_projected_ensemble(S.to_mps(c100), 0, M, seed=1)
print(f"MPS sampling: {M} samples in {time.perf_counter() - t0:.2f} s")
m_s = S.hamming_weights(bits)
fid = np.abs(np.sum(chi_s.conj() * chi_ex[m_s], axis=1))**2
log_mult = gammaln(n) - gammaln(m_s + 1) - gammaln(n - m_s)          # log C(n-1, m)
print(f"min fidelity, sampled vs exact conditional states: {fid.min():.12f}")
print(f"max relative error in Born probabilities: {np.abs(p_s * np.exp(log_mult) / w_ex[m_s] - 1).max():.1e}")

freq = np.bincount(m_s, minlength=n) / M
plt.figure(figsize=(5.5, 3.3))
plt.bar(np.arange(n), freq, width=1.0, alpha=0.5, label=f"MPS samples (M={M})")
plt.plot(np.arange(n), w_ex, "k-", lw=1.2, label=r"exact $w_m$")
plt.xlim(0, n); plt.xlabel("m (ones measured in environment)"); plt.ylabel("probability")
plt.legend(); plt.tight_layout(); plt.show()

## 5. GQS at 500 qubits: regular vs. chaotic

The initial Bloch vector $(\sin\theta\cos\phi,\ \sin\theta\sin\phi,\ \cos\theta)$ lies in the $y$–$z$ plane, so we plot the $y$–$z$ projection and colour points by $x$. Marker area is proportional to the class weight $w_m$.

In [ ]:
n = 500
kicks_show = [0, 1, 5, 20, 100]
fig, axes = plt.subplots(2, len(kicks_show), figsize=(3 * len(kicks_show), 6.2), sharex=True, sharey=True)
for row, kap in enumerate([0.5, 2.5]):
    U = S.floquet_operator(n, kap); ck = S.coherent_state(n, theta, phi); done = 0
    for ax, kk in zip(axes[row], kicks_show):
        for _ in range(kk - done):
            ck = U @ ck
        done = kk
        chi_m, w_m = S.projected_ensemble(ck)
        keep = w_m > 1e-8
        b = bloch_vectors(chi_m[keep])
        ax.add_patch(plt.Circle((0, 0), 1, fill=False, color="0.7"))
        sc = ax.scatter(b[:, 1], b[:, 2], s=3000 * w_m[keep], c=b[:, 0], cmap="coolwarm", vmin=-1, vmax=1, alpha=0.8)
        ax.set_aspect("equal"); ax.set_xlim(-1.1, 1.1); ax.set_ylim(-1.1, 1.1)
        ax.set_title(fr"$\kappa$={kap}, kick {kk}", fontsize=10)
for ax in axes[1]: ax.set_xlabel("y")
for ax in axes[:, 0]: ax.set_ylabel("z")
fig.colorbar(sc, ax=axes, shrink=0.6, label="x")
plt.suptitle(f"Single-qubit GQS of the kicked top, n = {n}", y=0.98)
plt.show()

## 6. Wasserstein sensitivity vs. system size (up to 500 qubits)

This is the same $W_1(t)/W_1(0)$ diagnostic as Section 3 and the paper. With full-space code it was limited to $n\le6$ in notebook 04. Here we go to $n=500$. The second panel compares even $n$ (integer total spin $j$) with odd $n$ (half-integer $j$).

In [ ]:
if ot is not None:
    K = 200
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
    for n in [6, 20, 100, 500]:
        ax[0].plot(sensitivity_curve(n, kappa, K), lw=1, label=f"n={n}")
    ax[0].set_xlabel("kicks"); ax[0].set_ylabel(r"$W_1(t)/W_1(0)$"); ax[0].set_title(fr"$\kappa$={kappa}")
    ax[0].legend()

    ns = np.arange(20, 61)
    avg = [sensitivity_curve(n, kappa, K)[K // 2:].mean() for n in ns]
    ax[1].plot(ns[ns % 2 == 0], np.array(avg)[ns % 2 == 0], "o-", label="even n (integer j)")
    ax[1].plot(ns[ns % 2 == 1], np.array(avg)[ns % 2 == 1], "s-", label="odd n (half-integer j)")
    ax[1].set_xlabel("n"); ax[1].set_ylabel(r"late-time mean $W_1(t)/W_1(0)$"); ax[1].legend()
    plt.tight_layout(); plt.show()

## Takeaways

- **Symmetry first, tensor networks second.** For the kicked top, the Dicke subspace gives exact dynamics and exact projected ensembles at $n\sim10^3$ in under a second. The full-space code stops at $n\approx12$–$14$.
- **The MPS view explains why it's easy.** Permutation symmetry caps the bond dimension at $n/2+1$, so entanglement grows at most like $\log_2 n$. An MPS reproduces the exact ensemble at 100 qubits.
- **Diagnostics scale.** The paper's Wasserstein sensitivity diagnostic, previously limited to $n\le6$, now runs at $n=500$.

## Next steps
- Break the symmetry, for example with site-dependent kicks, disorder, or local noise trajectories. Then the Dicke trick fails, and the MPS with swap networks (or sampling from notebook 06) becomes necessary.
- Compare with `gqs.qiskit_qkt` finite-shot conditional tomography at small $n$, to estimate how many shots a hardware run would need.
- Compute $\Gamma$ (separation rates) at large $n$ by swapping `sensitivity_curve` into `Gamma_calculator`'s trajectory loop.